In [1]:
import json

In [3]:
with open('data/top_artists_with_metrics.json') as f:
    data = json.load(f)

In [4]:
d = {}
for row in data:
    if row['mbid'] not in d:
        d[row['mbid']] = (row['listeners'], row['playcount'])
    

In [6]:
import pandas as pd

In [7]:
df = pd.read_csv('graphs/initial_graph/node_features.csv')

In [28]:
df = df.drop_duplicates(subset=['mbid'])

In [29]:
df.shape

(2806, 12)

In [30]:
mbids = df['mbid'].tolist()

In [31]:
len([mbid for mbid in mbids if mbid in d])

770

In [32]:
from tqdm import tqdm

In [33]:
import requests
import time
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("LASTFM_API_KEY")
def get_artist_metrics(mbids: list[str], sleep_time: float = 0.05):
    """
    Retrieves Last.fm metrics (listeners and playcount) for a list of artist MBIDs.

    Parameters:
        api_key (str): Your Last.fm API key.
        mbids (list[str]): A list of MusicBrainz IDs (MBIDs) for artists.
        sleep_time (float): Delay between API calls to avoid rate limiting (default: 0.25s).

    Returns:
        dict: {mbid: (listeners, playcount)} — both values are ints or None if unavailable.
    """
    base_url = "https://ws.audioscrobbler.com/2.0/"
    results = {}

    for mbid in tqdm(mbids):
        params = {
            "method": "artist.getInfo",
            "api_key": api_key,
            "mbid": mbid,
            "format": "json"
        }
        try:
            response = requests.get(base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            # Handle missing or malformed data gracefully
            artist = data.get("artist", {})
            listeners = playcount = None

            if artist:
                listeners = int(artist.get("stats", {}).get("listeners", 0)) if artist.get("stats", {}).get("listeners") else None
                playcount = int(artist.get("stats", {}).get("playcount", 0)) if artist.get("stats", {}).get("playcount") else None

            results[mbid] = (listeners, playcount)

        except Exception as e:
            # Record failure gracefully with None
            results[mbid] = (None, None)
            print(f"Failed to fetch metrics for MBID {mbid}: {e}")

        time.sleep(sleep_time)  # avoid hitting rate limits

    return results


In [34]:
mbids_with_metrics = get_artist_metrics(mbids)

 58%|█████▊    | 1622/2806 [25:07<34:16,  1.74s/it] 

Failed to fetch metrics for MBID 679ec43c-bc04-4bb0-b619-2da702a214ba: Expecting value: line 1 column 1 (char 0)


100%|██████████| 2806/2806 [42:11<00:00,  1.11it/s]


In [38]:
for mbid in mbids_with_metrics:
    if None in mbids_with_metrics[mbid]:
        mbids_with_metrics[mbid] = (0, 0)

In [40]:
final = [{'mbid': mbid, 'listeners': mbids_with_metrics[mbid][0], 'playcount': mbids_with_metrics[mbid][1]} for mbid in mbids_with_metrics]

In [44]:
pd.DataFrame(final).to_csv('expanded_artist_metrics.csv', index=False)